In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.cell import range_boundaries

def read_excel_table(path: str, table_name: str, sheet_name: str | None = None, *, data_only: bool = True) -> pd.DataFrame:
    wb = load_workbook(path, data_only=data_only)
    ws = wb[sheet_name] if sheet_name else wb.active

    tables = {t.name: t for t in ws.tables.values()}
    if table_name not in tables:
        raise KeyError(f"Table '{table_name}' not found on sheet '{ws.title}'. Available: {list(tables)}")

    ref = tables[table_name].ref  # e.g. "A1:H20"
    min_col, min_row, max_col, max_row = range_boundaries(ref)

    # pull values
    values = [
        list(r) for r in ws.iter_rows(
            min_row=min_row, max_row=max_row,
            min_col=min_col, max_col=max_col,
            values_only=True
        )
    ]

    if not values or len(values) < 2:
        return pd.DataFrame()

    headers = list(values[0])

    # fix blank headers (common in messy sheets)
    headers = [
        (str(h).strip() if h is not None and str(h).strip() != "" else f"Unnamed_{i}")
        for i, h in enumerate(headers, start=1)
    ]

    df = pd.DataFrame(values[1:], columns=headers)

    # drop fully empty rows/cols
    df = df.dropna(how="all").dropna(axis=1, how="all")

    # strip whitespace for string cells
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    return df

# ---- usage ----
path = r"C:\Users\vance\OneDrive\3) References\1) Technical References\F) Spreadsheets\Load combinations.xlsx"
sheet_name = "Sheet2"
table_name = "lc_table_no_seismic"

df = read_excel_table(path, table_name, sheet_name)
df


KeyboardInterrupt: 

In [205]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 26 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   NOTES     51 non-null     object 
 1   S/N       51 non-null     int64  
 2   LC        51 non-null     object 
 3   DL        51 non-null     float64
 4   SDL       51 non-null     float64
 5   LL        51 non-null     float64
 6   GIFDLX+   51 non-null     float64
 7   GIFDLX-   51 non-null     float64
 8   GIFDLY+   51 non-null     float64
 9   GIFDLY-   51 non-null     float64
 10  GIFSDLX+  51 non-null     float64
 11  GIFSDLX-  51 non-null     float64
 12  GIFSDLY+  51 non-null     float64
 13  GIFSDLY-  51 non-null     float64
 14  GIFLLX+   51 non-null     float64
 15  GIFLLX-   51 non-null     float64
 16  GIFLLY+   51 non-null     float64
 17  GIFLLY-   51 non-null     float64
 18  WINDX+    51 non-null     float64
 19  WINDX-    51 non-null     float64
 20  WINDY+    51 non-null     float64


In [206]:
PF_start_col = "DL"
#getting the column index of a start of PF columns PF_start_col
PF_start_idx = df.columns.get_loc(PF_start_col)
df.iloc[:, PF_start_idx:]


,DL,SDL,LL,GIFDLX+,GIFDLX-,GIFDLY+,GIFDLY-,GIFSDLX+,GIFSDLX-,GIFSDLY+,...,GIFLLY+,GIFLLY-,WINDX+,WINDX-,WINDY+,WINDY-,NLX+,NLX-,NLY+,NLY-
0,1.35,1.35,1.50,1.35,0.00,0.00,0.00,1.35,0.00,0.00,...,0.00,0.00,0.75,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,1.35,1.35,1.50,0.00,1.35,0.00,0.00,0.00,1.35,0.00,...,0.00,0.00,0.00,0.75,0.00,0.00,0.00,0.00,0.00,0.00
2,1.35,1.35,1.50,0.00,0.00,1.35,0.00,0.00,0.00,1.35,...,1.50,0.00,0.00,0.00,0.75,0.00,0.00,0.00,0.00,0.00
3,1.35,1.35,1.50,0.00,0.00,0.00,1.35,0.00,0.00,0.00,...,0.00,1.50,0.00,0.00,0.00,0.75,0.00,0.00,0.00,0.00
4,1.35,1.35,1.05,1.35,0.00,0.00,0.00,1.35,0.00,0.00,...,0.00,0.00,1.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,1.35,1.35,1.05,0.00,1.35,0.00,0.00,0.00,1.35,0.00,...,0.00,0.00,0.00,1.50,0.00,0.00,0.00,0.00,0.00,0.00
6,1.35,1.35,1.05,0.00,0.00,1.35,0.00,0.00,0.00,1.35,...,1.05,0.00,0.00,0.00,1.50,0.00,0.00,0.00,0.00,0.00
7,1.35,1.35,1.05,0.00,0.00,0.00,1.35,0.00,0.00,0.00,...,0.00,1.05,0.00,0.00,0.00,1.50,0.00,0.00,0.00,0.00
8,1.00,1.00,1.30,1.00,0.00,0.00,0.00,1.00,0.00,0.00,...,0.00,0.00,0.65,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,1.00,1.00,1.30,0.00,1.00,0.00,0.00,0.00,1.00,0.00,...,0.00,0.00,0.00,0.65,0.00,0.00,0.00,0.00,0.00,0.00


In [207]:
df_LC = df.iloc[:, PF_start_idx-1:]
df_LC

,LC,DL,SDL,LL,GIFDLX+,GIFDLX-,GIFDLY+,GIFDLY-,GIFSDLX+,GIFSDLX-,...,GIFLLY+,GIFLLY-,WINDX+,WINDX-,WINDY+,WINDY-,NLX+,NLX-,NLY+,NLY-
0,DA1-1-LLX+,1.35,1.35,1.50,1.35,0.00,0.00,0.00,1.35,0.00,...,0.00,0.00,0.75,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,DA1-1-LLX-,1.35,1.35,1.50,0.00,1.35,0.00,0.00,0.00,1.35,...,0.00,0.00,0.00,0.75,0.00,0.00,0.00,0.00,0.00,0.00
2,DA1-1-LLY+,1.35,1.35,1.50,0.00,0.00,1.35,0.00,0.00,0.00,...,1.50,0.00,0.00,0.00,0.75,0.00,0.00,0.00,0.00,0.00
3,DA1-1-LLY-,1.35,1.35,1.50,0.00,0.00,0.00,1.35,0.00,0.00,...,0.00,1.50,0.00,0.00,0.00,0.75,0.00,0.00,0.00,0.00
4,DA1-1-WLX+,1.35,1.35,1.05,1.35,0.00,0.00,0.00,1.35,0.00,...,0.00,0.00,1.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,DA1-1-WLX-,1.35,1.35,1.05,0.00,1.35,0.00,0.00,0.00,1.35,...,0.00,0.00,0.00,1.50,0.00,0.00,0.00,0.00,0.00,0.00
6,DA1-1-WLY+,1.35,1.35,1.05,0.00,0.00,1.35,0.00,0.00,0.00,...,1.05,0.00,0.00,0.00,1.50,0.00,0.00,0.00,0.00,0.00
7,DA1-1-WLY-,1.35,1.35,1.05,0.00,0.00,0.00,1.35,0.00,0.00,...,0.00,1.05,0.00,0.00,0.00,1.50,0.00,0.00,0.00,0.00
8,DA1-2-LLX+,1.00,1.00,1.30,1.00,0.00,0.00,0.00,1.00,0.00,...,0.00,0.00,0.65,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,DA1-2-LLX-,1.00,1.00,1.30,0.00,1.00,0.00,0.00,0.00,1.00,...,0.00,0.00,0.00,0.65,0.00,0.00,0.00,0.00,0.00,0.00


In [208]:
import pandas as pd

# df = your current dataframe
# If you read from Excel:
# df = pd.read_excel("your_file.xlsx", sheet_name="...", dtype=str)  # optional

# Make sure LC exists (adjust if your combo name column is different)
id_col = "LC"

# Melt everything except LC into rows
long_df = (
    df_LC.melt(
        id_vars=[id_col],
        var_name="LoadCase",
        value_name="Factor"
    )
)

# Optional: convert Factor to numeric and drop blanks/zeros
long_df["Factor"] = pd.to_numeric(long_df["Factor"], errors="coerce")
long_df = long_df.dropna(subset=["Factor"])
long_df = long_df[long_df["Factor"] != 0]

# # Optional: nicer ordering
# long_df = long_df.sort_values([id_col, "LoadCase"]).reset_index(drop=True)

# long_df


In [209]:
long_df

,LC,LoadCase,Factor
0,DA1-1-LLX+,DL,1.35
1,DA1-1-LLX-,DL,1.35
2,DA1-1-LLY+,DL,1.35
3,DA1-1-LLY-,DL,1.35
4,DA1-1-WLX+,DL,1.35
...,...,...,...
1152,DA1-1-LL.NLY-,NLY-,0.75
1156,DA1-1-NLY-,NLY-,1.50
1160,DA1-2-LL.NLY-,NLY-,0.65
1164,DA1-2-NLY-,NLY-,1.30


In [210]:
long_df["e2k_line"] = "COMBO" + ' "' + long_df[id_col].astype(str) + '" LOADCASE "' + long_df["LoadCase"].astype(str) + '" SF ' + long_df["Factor"].astype(str)
long_df["e2k_line"]

0            COMBO "DA1-1-LLX+" LOADCASE "DL" SF 1.35
1            COMBO "DA1-1-LLX-" LOADCASE "DL" SF 1.35
2            COMBO "DA1-1-LLY+" LOADCASE "DL" SF 1.35
3            COMBO "DA1-1-LLY-" LOADCASE "DL" SF 1.35
4            COMBO "DA1-1-WLX+" LOADCASE "DL" SF 1.35
                            ...                      
1152    COMBO "DA1-1-LL.NLY-" LOADCASE "NLY-" SF 0.75
1156        COMBO "DA1-1-NLY-" LOADCASE "NLY-" SF 1.5
1160    COMBO "DA1-2-LL.NLY-" LOADCASE "NLY-" SF 0.65
1164        COMBO "DA1-2-NLY-" LOADCASE "NLY-" SF 1.3
1172          COMBO "SLS-NLY-" LOADCASE "NLY-" SF 1.0
Name: e2k_line, Length: 345, dtype: object

In [211]:
# e2k LC headers
df['e2k_line'] = "COMBO" + ' "' + long_df[id_col].astype(str) + '" ' + 'TYPE "Linear Add" NOTES ' +'"' + df["NOTES"] +'"'
df['e2k_line']


0     COMBO "DA1-1-LLX+" TYPE "Linear Add" NOTES "DA...
1     COMBO "DA1-1-LLX-" TYPE "Linear Add" NOTES "DA...
2     COMBO "DA1-1-LLY+" TYPE "Linear Add" NOTES "DA...
3     COMBO "DA1-1-LLY-" TYPE "Linear Add" NOTES "DA...
4     COMBO "DA1-1-WLX+" TYPE "Linear Add" NOTES "DA...
5     COMBO "DA1-1-WLX-" TYPE "Linear Add" NOTES "DA...
6     COMBO "DA1-1-WLY+" TYPE "Linear Add" NOTES "DA...
7     COMBO "DA1-1-WLY-" TYPE "Linear Add" NOTES "DA...
8     COMBO "DA1-2-LLX+" TYPE "Linear Add" NOTES "DA...
9     COMBO "DA1-2-LLX-" TYPE "Linear Add" NOTES "DA...
10    COMBO "DA1-2-LLY+" TYPE "Linear Add" NOTES "DA...
11    COMBO "DA1-2-LLY-" TYPE "Linear Add" NOTES "DA...
12    COMBO "DA1-2-WLX+" TYPE "Linear Add" NOTES "DA...
13    COMBO "DA1-2-WLX-" TYPE "Linear Add" NOTES "DA...
14    COMBO "DA1-2-WLY+" TYPE "Linear Add" NOTES "DA...
15    COMBO "DA1-2-WLY-" TYPE "Linear Add" NOTES "DA...
16    COMBO "SLS-LLX+" TYPE "Linear Add" NOTES "Serv...
17    COMBO "SLS-LLX-" TYPE "Linear Add" NOTES "

In [212]:
df

,NOTES,S/N,LC,DL,SDL,LL,GIFDLX+,GIFDLX-,GIFDLY+,GIFDLY-,...,GIFLLY-,WINDX+,WINDX-,WINDY+,WINDY-,NLX+,NLX-,NLY+,NLY-,e2k_line
0,DA1-1 lateral loads considered LL leading,1,DA1-1-LLX+,1.35,1.35,1.50,1.35,0.00,0.00,0.00,...,0.00,0.75,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"COMBO ""DA1-1-LLX+"" TYPE ""Linear Add"" NOTES ""DA..."
1,DA1-1 lateral loads considered LL leading,2,DA1-1-LLX-,1.35,1.35,1.50,0.00,1.35,0.00,0.00,...,0.00,0.00,0.75,0.00,0.00,0.00,0.00,0.00,0.00,"COMBO ""DA1-1-LLX-"" TYPE ""Linear Add"" NOTES ""DA..."
2,DA1-1 lateral loads considered LL leading,3,DA1-1-LLY+,1.35,1.35,1.50,0.00,0.00,1.35,0.00,...,0.00,0.00,0.00,0.75,0.00,0.00,0.00,0.00,0.00,"COMBO ""DA1-1-LLY+"" TYPE ""Linear Add"" NOTES ""DA..."
3,DA1-1 lateral loads considered LL leading,4,DA1-1-LLY-,1.35,1.35,1.50,0.00,0.00,0.00,1.35,...,1.50,0.00,0.00,0.00,0.75,0.00,0.00,0.00,0.00,"COMBO ""DA1-1-LLY-"" TYPE ""Linear Add"" NOTES ""DA..."
4,DA1-1 lateral loads considered WL leading,5,DA1-1-WLX+,1.35,1.35,1.05,1.35,0.00,0.00,0.00,...,0.00,1.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"COMBO ""DA1-1-WLX+"" TYPE ""Linear Add"" NOTES ""DA..."
5,DA1-1 lateral loads considered WL leading,6,DA1-1-WLX-,1.35,1.35,1.05,0.00,1.35,0.00,0.00,...,0.00,0.00,1.50,0.00,0.00,0.00,0.00,0.00,0.00,"COMBO ""DA1-1-WLX-"" TYPE ""Linear Add"" NOTES ""DA..."
6,DA1-1 lateral loads considered WL leading,7,DA1-1-WLY+,1.35,1.35,1.05,0.00,0.00,1.35,0.00,...,0.00,0.00,0.00,1.50,0.00,0.00,0.00,0.00,0.00,"COMBO ""DA1-1-WLY+"" TYPE ""Linear Add"" NOTES ""DA..."
7,DA1-1 lateral loads considered WL leading,8,DA1-1-WLY-,1.35,1.35,1.05,0.00,0.00,0.00,1.35,...,1.05,0.00,0.00,0.00,1.50,0.00,0.00,0.00,0.00,"COMBO ""DA1-1-WLY-"" TYPE ""Linear Add"" NOTES ""DA..."
8,DA1-2 lateral loads considered LL leading,9,DA1-2-LLX+,1.00,1.00,1.30,1.00,0.00,0.00,0.00,...,0.00,0.65,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"COMBO ""DA1-2-LLX+"" TYPE ""Linear Add"" NOTES ""DA..."
9,DA1-2 lateral loads considered LL leading,10,DA1-2-LLX-,1.00,1.00,1.30,0.00,1.00,0.00,0.00,...,0.00,0.00,0.65,0.00,0.00,0.00,0.00,0.00,0.00,"COMBO ""DA1-2-LLX-"" TYPE ""Linear Add"" NOTES ""DA..."


In [213]:
#add in the envelope ULT load combinations

#filter rows containing DA1- under column "LC"
df_env_comb = df.loc[df["LC"].str.contains("DA1-")].drop_duplicates(subset=["LC"])
#drop rows which are "DA1-1(Grav)" and "DA1-2(Grav)"
df_env_comb = df_env_comb[~df_env_comb["LC"].isin(["DA1-1(Grav)","DA1-2(Grav)"])]

DA1_1_lines = df_env_comb.loc[df_env_comb["LC"].str.contains("DA1-1")]
DA1_2_lines = df_env_comb.loc[df_env_comb["LC"].str.contains("DA1-2")]


DA1_1_lines.loc[:, 'e2k_line'] = 'COMBO "DA1-1"  LOADCOMBO "' + DA1_1_lines["LC"].astype(str) + '" SF 1' 
DA1_2_lines.loc[:, 'e2k_line'] = 'COMBO "DA1-2"  LOADCOMBO "' + DA1_2_lines["LC"].astype(str) + '" SF 1' 




In [214]:
#add in the envelope SERV combinations

#filter rows containing SLS- under column "LC"
SLS_lines = df.loc[df["LC"].str.contains("SLS-")].drop_duplicates(subset=["LC"])
#drop rows which are "SLS-1(Grav)" and "SLS-2(Grav)"
SLS_lines = df.loc[df["LC"].str.contains("SLS-")]
SLS_lines.loc[:, 'e2k_line'] = 'COMBO "SERV"  LOADCOMBO "' + SLS_lines["LC"].astype(str) + '" SF 1' 


In [215]:
# LC headers for envelopes
envelope_header_rows = pd.DataFrame({
    'e2k_line': ['COMBO "DA1-1" TYPE "Envelope"', 'COMBO "DA1-2" TYPE "Envelope"','COMBO "SERV" TYPE "Envelope"']
})


In [216]:
envelope_header_rows

,e2k_line
0,"COMBO ""DA1-1"" TYPE ""Envelope"""
1,"COMBO ""DA1-2"" TYPE ""Envelope"""
2,"COMBO ""SERV"" TYPE ""Envelope"""


In [217]:
SLS_lines

,NOTES,S/N,LC,DL,SDL,LL,GIFDLX+,GIFDLX-,GIFDLY+,GIFDLY-,...,GIFLLY-,WINDX+,WINDX-,WINDY+,WINDY-,NLX+,NLX-,NLY+,NLY-,e2k_line
16,Service load LL leading,17,SLS-LLX+,1.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.75,0.00,0.00,0.00,0.0,0.0,0.0,0.0,"COMBO ""SERV"" LOADCOMBO ""SLS-LLX+"" SF 1"
17,Service load LL leading,18,SLS-LLX-,1.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.00,0.75,0.00,0.00,0.0,0.0,0.0,0.0,"COMBO ""SERV"" LOADCOMBO ""SLS-LLX-"" SF 1"
18,Service load LL leading,19,SLS-LLY+,1.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.00,0.00,0.75,0.00,0.0,0.0,0.0,0.0,"COMBO ""SERV"" LOADCOMBO ""SLS-LLY+"" SF 1"
19,Service load LL leading,20,SLS-LLY-,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.00,0.00,0.00,0.75,0.0,0.0,0.0,0.0,"COMBO ""SERV"" LOADCOMBO ""SLS-LLY-"" SF 1"
20,Service load WL leading,21,SLS-WLX+,1.0,1.0,0.7,1.0,0.0,0.0,0.0,...,0.0,1.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,"COMBO ""SERV"" LOADCOMBO ""SLS-WLX+"" SF 1"
21,Service load WL leading,22,SLS-WLX-,1.0,1.0,0.7,0.0,1.0,0.0,0.0,...,0.0,0.00,1.00,0.00,0.00,0.0,0.0,0.0,0.0,"COMBO ""SERV"" LOADCOMBO ""SLS-WLX-"" SF 1"
22,Service load WL leading,23,SLS-WLY+,1.0,1.0,0.7,0.0,0.0,1.0,0.0,...,0.0,0.00,0.00,1.00,0.00,0.0,0.0,0.0,0.0,"COMBO ""SERV"" LOADCOMBO ""SLS-WLY+"" SF 1"
23,Service load WL leading,24,SLS-WLY-,1.0,1.0,0.7,0.0,0.0,0.0,1.0,...,0.7,0.00,0.00,0.00,1.00,0.0,0.0,0.0,0.0,"COMBO ""SERV"" LOADCOMBO ""SLS-WLY-"" SF 1"
43,Service load(NL replace WL) LL leading,44,SLS-LLX.NL+,1.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.75,0.00,0.00,0.00,0.0,0.0,0.0,0.0,"COMBO ""SERV"" LOADCOMBO ""SLS-LLX.NL+"" SF 1"
44,Service load(NL replace WL) LL leading,45,SLS-LLX.NL-,1.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.00,0.75,0.00,0.00,0.0,0.0,0.0,0.0,"COMBO ""SERV"" LOADCOMBO ""SLS-LLX.NL-"" SF 1"


In [218]:
import pandas as pd

long_df = pd.concat([long_df, df[["e2k_line"]],DA1_1_lines[["e2k_line"]],DA1_2_lines[["e2k_line"]],SLS_lines[["e2k_line"]],envelope_header_rows], ignore_index=True)

long_df.sort_values(by="e2k_line",inplace=True,ascending=False)


In [219]:
long_df

,LC,LoadCase,Factor,e2k_line
371,NaN,NaN,NaN,"COMBO ""WL"" TYPE ""Linear Add"" NOTES ""Gk+Qk"""
77,WL,SDL,1.0,"COMBO ""WL"" LOADCASE ""SDL"" SF 1.0"
128,WL,LL,1.0,"COMBO ""WL"" LOADCASE ""LL"" SF 1.0"
26,WL,DL,1.0,"COMBO ""WL"" LOADCASE ""DL"" SF 1.0"
368,NaN,NaN,NaN,"COMBO ""SLS-WLY-"" TYPE ""Linear Add"" NOTES ""Serv..."
...,...,...,...,...
396,NaN,NaN,NaN,"COMBO ""DA1-1"" LOADCOMBO ""DA1-1-LLX+"" SF 1"
407,NaN,NaN,NaN,"COMBO ""DA1-1"" LOADCOMBO ""DA1-1-LL.NLY-"" SF 1"
406,NaN,NaN,NaN,"COMBO ""DA1-1"" LOADCOMBO ""DA1-1-LL.NLY+"" SF 1"
405,NaN,NaN,NaN,"COMBO ""DA1-1"" LOADCOMBO ""DA1-1-LL.NLX-"" SF 1"


In [221]:
long_df['e2k_line'].to_csv(r"C:\Users\vance\OneDrive\3) References\1) Technical References\E) Modelling\ETabs\Templates\LC_e2k.csv", index=False, header=False)

In [ ]:
# Add envelope load combinations
